# 14 sklearn Pipeline 集成示例

覆盖 Pipeline / ColumnTransformer / VotingClassifier / StackingClassifier 与 hscredit 组件的结合使用。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression as SKLearnLR
from sklearn.preprocessing import StandardScaler
import hscredit as hsc
from hscredit import (germancredit, init_setting, seed_everything,
    Pipeline, make_pipeline, ColumnTransformer, make_column_selector,
    make_column_transformer, FunctionTransformer,
    VotingClassifier, StackingClassifier,
    WOEEncoder, TargetEncoder, OptimalBinning, IVSelector,
    LogisticRegression,
)
from hscredit.core.metrics import ks, auc
seed_everything(42); init_setting()
df = germancredit()
target = 'creditability'
X = df.drop(columns=[target]); y = df[target]
num_feats = X.select_dtypes('number').columns.tolist()
cat_feats = X.select_dtypes('object').columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f'数值特征: {len(num_feats)}, 类别特征: {len(cat_feats)}')

## 1. 简单 Pipeline

In [ ]:
# WOE编码 + 逻辑回归
pipe_lr = Pipeline([
    ('woe', WOEEncoder(max_n_bins=5)),
    ('lr',  LogisticRegression(C=1.0, max_iter=1000)),
])
pipe_lr.fit(X_tr[num_feats], y_tr)
prob = pipe_lr.predict_proba(X_te[num_feats])[:,1]
print(f'WOE+LR  KS:{ks(y_te,prob):.4f}  AUC:{auc(y_te,prob):.4f}')

In [ ]:
# TargetEncoding + RF
pipe_rf = Pipeline([
    ('te', TargetEncoder()),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
])
pipe_rf.fit(X_tr, y_tr)
prob_rf = pipe_rf.predict_proba(X_te)[:,1]
print(f'TE+RF   KS:{ks(y_te,prob_rf):.4f}  AUC:{auc(y_te,prob_rf):.4f}')

## 2. ColumnTransformer — 数值 & 类别分开处理

In [ ]:
preprocessor = ColumnTransformer([
    ('num', WOEEncoder(max_n_bins=5), num_feats),
    ('cat', TargetEncoder(smoothing=5), cat_feats),
])
pipe_col = Pipeline([
    ('prep', preprocessor),
    ('lr',   SKLearnLR(C=1.0, max_iter=1000)),
])
pipe_col.fit(X_tr, y_tr)
prob_col = pipe_col.predict_proba(X_te)[:,1]
print(f'ColumnTransformer+LR  KS:{ks(y_te,prob_col):.4f}  AUC:{auc(y_te,prob_col):.4f}')

## 3. 含特征筛选的 Pipeline

In [ ]:
pipe_sel = Pipeline([
    ('woe', WOEEncoder(max_n_bins=5)),
    ('iv',  IVSelector(threshold=0.02, max_n_bins=5)),
    ('lr',  LogisticRegression(C=1.0, max_iter=1000)),
])
pipe_sel.fit(X_tr[num_feats], y_tr)
prob_sel = pipe_sel.predict_proba(X_te[num_feats])[:,1]
print(f'WOE+IV筛选+LR  KS:{ks(y_te,prob_sel):.4f}  AUC:{auc(y_te,prob_sel):.4f}')
print('筛选后特征:', pipe_sel.named_steps['iv'].get_feature_names_out().tolist())

## 4. VotingClassifier — 软投票集成

In [ ]:
woe_enc = WOEEncoder(max_n_bins=5)
X_woe_tr = woe_enc.fit_transform(X_tr[num_feats], y_tr)
X_woe_te = woe_enc.transform(X_te[num_feats])

voting = VotingClassifier(estimators=[
    ('lr',  SKLearnLR(C=1.0, max_iter=1000)),
    ('gbm', GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ('rf',  RandomForestClassifier(n_estimators=50, random_state=42)),
], voting='soft', weights=[2, 1, 1])
voting.fit(X_woe_tr, y_tr)
prob_vote = voting.predict_proba(X_woe_te)[:,1]
print(f'VotingClassifier  KS:{ks(y_te,prob_vote):.4f}  AUC:{auc(y_te,prob_vote):.4f}')

## 5. StackingClassifier — Stacking集成

In [ ]:
stacking = StackingClassifier(
    estimators=[
        ('gbm', GradientBoostingClassifier(n_estimators=50, random_state=42)),
        ('rf',  RandomForestClassifier(n_estimators=50, random_state=42)),
    ],
    final_estimator=SKLearnLR(C=1.0, max_iter=1000),
    cv=3,
    passthrough=False,
)
stacking.fit(X_woe_tr, y_tr)
prob_stack = stacking.predict_proba(X_woe_te)[:,1]
print(f'StackingClassifier  KS:{ks(y_te,prob_stack):.4f}  AUC:{auc(y_te,prob_stack):.4f}')

## 6. cross_val_score — 交叉验证

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(pipe_lr, X[num_feats], y, cv=5, scoring='roc_auc')
print(f'5折CV AUC: {scores.round(4)}')
print(f'均值: {scores.mean():.4f} ± {scores.std():.4f}')

## 7. FunctionTransformer — 自定义转换步骤

In [ ]:
import numpy as np
def log_transform(X):
    return np.log1p(np.abs(X.values if hasattr(X,'values') else X))

pipe_log = Pipeline([
    ('log', FunctionTransformer(log_transform)),
    ('scl', StandardScaler()),
    ('lr',  SKLearnLR(C=1.0, max_iter=1000)),
])
pipe_log.fit(X_tr[num_feats], y_tr)
prob_log = pipe_log.predict_proba(X_te[num_feats])[:,1]
print(f'log+scale+LR  KS:{ks(y_te,prob_log):.4f}  AUC:{auc(y_te,prob_log):.4f}')

## 8. 模型指标汇总对比

In [ ]:
rows = [
    {'模型': 'WOE+LR',         'KS': ks(y_te,prob),       'AUC': auc(y_te,prob)},
    {'模型': 'TE+RF',          'KS': ks(y_te,prob_rf),    'AUC': auc(y_te,prob_rf)},
    {'模型': 'CT+LR',          'KS': ks(y_te,prob_col),   'AUC': auc(y_te,prob_col)},
    {'模型': 'WOE+IV+LR',      'KS': ks(y_te,prob_sel),   'AUC': auc(y_te,prob_sel)},
    {'模型': 'VotingCls',      'KS': ks(y_te,prob_vote),  'AUC': auc(y_te,prob_vote)},
    {'模型': 'StackingCls',    'KS': ks(y_te,prob_stack), 'AUC': auc(y_te,prob_stack)},
    {'模型': 'log+scale+LR',   'KS': ks(y_te,prob_log),   'AUC': auc(y_te,prob_log)},
]
pd.DataFrame(rows).sort_values('KS', ascending=False).round(4)